# **YOLO V8 Model Training**




# **Step 1 : Dataset Loading**

In [25]:
import kagglehub

path = kagglehub.dataset_download("redzapdos123/indian-driving-dataset-detections-yolov11")

print("Path to dataset files:", path)


100%|██████████| 20.5G/20.5G [03:21<00:00, 109MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1


# **Step 2 : Dependencies Installation**

In [26]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.7 MB/s eta 0:00:00


# **Step 3 : YAML File Upload**

In [34]:
from google.colab import files
import os

# Upload file
uploaded = files.upload()

for filename in uploaded.keys():
    file_path = f"/content/{filename}"

    # Check file extension
    if filename.lower().endswith(('.yaml', '.yml')):
        print(f"✅ '{filename}' is a YAML file")
        print("File path:", file_path)
    else:
        print(f"❌ '{filename}' is NOT a YAML file")


Saving new_data.yaml to new_data.yaml
✅ 'new_data.yaml' is a YAML file
File path: /content/new_data.yaml


# **Step 4 : Dataset / Directory Inspection**

In [30]:
import os

# Define the path where the dataset was downloaded by kagglehub
dataset_base_path = "/root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1"

print(f"Inspecting directory structure at: {dataset_base_path}")

# List contents of the base directory
print(f"\nContents of {dataset_base_path}:")
for item in os.listdir(dataset_base_path):
    print(item)

# Subdirectory Inspection
dataset_subdir = os.path.join(dataset_base_path, "IDDDetectionsYOLODataset")
if os.path.exists(dataset_subdir):
    print(f"\nContents of {dataset_subdir}:")
    for item in os.listdir(dataset_subdir):
        print(item)
    # Inspecting train, val & test directory by appending 1st 5 items
    expected_subdirs = ['train', 'val', 'test']
    for subdir in expected_subdirs:
        images_path = os.path.join(dataset_subdir, subdir, 'images')
        labels_path = os.path.join(dataset_subdir, subdir, 'labels')
        if os.path.exists(images_path):
            print(f"\nContents of {images_path} (first 5 items):")
            try:
                for i, item in enumerate(os.listdir(images_path)):
                    if i < 5:
                        print(item)
                    else:
                        break
            except Exception as e:
                print(f"Could not list contents: {e}")
        else:
            print(f"\n{images_path} not found.")

        if os.path.exists(labels_path):
            print(f"\nContents of {labels_path} (first 5 items):")
            try:
                for i, item in enumerate(os.listdir(labels_path)):
                    if i < 5:
                        print(item)
                    else:
                        break
            except Exception as e:
                print(f"Could not list contents: {e}")
        else:
            print(f"\n{labels_path} not found.")

Inspecting directory structure at: /root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1

Contents of /root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1:
IDDDetectionsYOLODataset

Contents of /root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset:
test
train
val
ReadMe.md
data.yaml
license.md

Contents of /root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset/train/images (first 5 items):
BLR-2018-06-20-07-01-47_part_29_0000224.jpg
BLR-2018-03-22_17-39-26_2_sideLeft_000012_r.jpg
BLR-2018-05-22_12-04-54_frontNear_001018_r.jpg
HYD-2018-08-24_13-22-50_0004964.jpg
BLR-2018-05-17_16-39-05_sideRight_000012_r.jpg

Contents of /root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset/train/labels (first 5 i

# **Step 5 : Model Configuration & Training**

### **No. of Epochs : 5 Epoch**

In [35]:
from ultralytics import YOLO
import os
import shutil
from google.colab import files

# Update Data.yaml path
data_yaml_path = '/content/new_data.yaml'

# Define the dataset directory path
dataset_base_path = "/root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset"

# Initialize the YOLO model
model = YOLO('yolov8m.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 5, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.005,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.1,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 0.05,
    'cls': 0.5,
    'dfl': 1.0,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 10, # Disable mosaic for the last 10 epochs
}

# Start training
results = model.train(**train_args)

print("Training complete.")

#Model Exporting
folder_path = "/content/runs/detect"
zip_path = "/content/5_epoch_training_output"

shutil.make_archive(zip_path, 'zip', folder_path)
files.download(zip_path + ".zip")

Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=0.05, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/new_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **No. of Epochs : 10 Epoch**

In [ ]:
from ultralytics import YOLO
import os
import shutil
from google.colab import files

# Update Data.yaml path
data_yaml_path = '/content/new_data.yaml'

# Define the dataset directory path
dataset_base_path = "/root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset"

# Initialize the YOLO model
model = YOLO('yolov8m.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 10, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.005,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.1,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 0.05,
    'cls': 0.5,
    'dfl': 1.0,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 10, # Disable mosaic for the last 10 epochs
}

# Start training
results = model.train(**train_args)

print("Training complete.")

#Model Exporting
folder_path = "/content/runs/detect"
zip_path = "/content/10_epoch_training_output"

shutil.make_archive(zip_path, 'zip', folder_path)
files.download(zip_path + ".zip")

### **No. of Epochs : 15 Epoch**

In [ ]:
from ultralytics import YOLO
import os
import shutil
from google.colab import files

# Update Data.yaml path
data_yaml_path = '/content/new_data.yaml'

# Define the dataset directory path
dataset_base_path = "/root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset"

# Initialize the YOLO model
model = YOLO('yolov8m.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 15, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.005,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.1,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 0.05,
    'cls': 0.5,
    'dfl': 1.0,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 10, # Disable mosaic for the last 10 epochs
}

# Start training
results = model.train(**train_args)

print("Training complete.")

#Model Exporting
folder_path = "/content/runs/detect"
zip_path = "/content/15_epoch_training_output"

shutil.make_archive(zip_path, 'zip', folder_path)
files.download(zip_path + ".zip")

### **No. of Epochs : 20 Epoch**

In [ ]:
from ultralytics import YOLO
import os
import shutil
from google.colab import files

# Update Data.yaml path
data_yaml_path = '/content/new_data.yaml'

# Define the dataset directory path
dataset_base_path = "/root/.cache/kagglehub/datasets/redzapdos123/indian-driving-dataset-detections-yolov11/versions/1/IDDDetectionsYOLODataset"

# Initialize the YOLO model
model = YOLO('yolov8m.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 20, # No. of Epoch
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.005,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.1,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 0.05,
    'cls': 0.5,
    'dfl': 1.0,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 10, # Disable mosaic for the last 10 epochs
}

# Start training
results = model.train(**train_args)

print("Training complete.")

#Model Exporting
folder_path = "/content/runs/detect"
zip_path = "/content/20_epoch_training_output"

shutil.make_archive(zip_path, 'zip', folder_path)
files.download(zip_path + ".zip")